# Gathering Relevant Data

In [1]:
# Base Directory
library(here)
BASE_DIR <- here()
print(BASE_DIR)

# Other dependencies
library(dplyr)

here() starts at /Users/amberteetsel/MSDS/STAT_5010/opioid-crisis



[1] "/Users/amberteetsel/MSDS/STAT_5010/opioid-crisis"



Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




## NCHS - Drug Poisoning Mortality by State

In [2]:
# Load Data
nchs_df <- read.csv(here("data", "NCHS_Mortality_Clean.csv"))

# Keep relevant columns
nchs_df = nchs_df[, c("state", "year", "sex", "age_group", "age_group_detail", "race", "population", "death_rate")]
head(nchs_df)
dim(nchs_df)

,state,year,sex,age_group,age_group_detail,race,population,death_rate
,<chr>,<int>,<chr>,<chr>,<chr>,<chr>,<int>,<dbl>
1,Alabama,1999,Both Sexes,All Ages,All Ages,All Races,4430143,3.8521
2,Alabama,2000,Both Sexes,All Ages,All Ages,All Races,4447100,4.4857
3,Alabama,2001,Both Sexes,All Ages,All Ages,All Races,4467634,4.8915
4,Alabama,2002,Both Sexes,All Ages,All Ages,All Races,4480089,4.7619
5,Alabama,2003,Both Sexes,All Ages,All Ages,All Races,4503491,4.4333
6,Alabama,2004,Both Sexes,All Ages,All Ages,All Races,4530729,6.3542


[1] 2862    8

In [3]:
# Filter for Entire US so we can use demographic breakdown
nchs_us <- nchs_df %>%
    filter(
        state == "United States",
        sex != "Both Sexes",
        age_group != "All Ages",
        race != "All Races",
        year >= 2000,
        state != "District of Columbia"
    )
nchs_us = nchs_us[, c("year", "sex", "age_group", "age_group_detail", "race", "death_rate")]

# Filter for State-Level Data
nchs_state <- nchs_df %>%
    filter(
        state != "United States",
        sex == "Both Sexes",
        age_group == "All Ages",
        race == "All Races",
        year >= 2000,
        state != "District of Columbia"
    )
nchs_state <- nchs_state[, c("state", "year", "death_rate")]

In [4]:
# Add state abbr to NCHS_State
nchs_state <- nchs_state %>%
    mutate(
        state_abbr = state.abb[match(toupper(state), toupper(state.name))]
    )
nchs_state = nchs_state[, c("state_abbr", "year", "death_rate")] %>%
    rename(state = state_abbr)
head(nchs_state)

,state,year,death_rate
,<chr>,<int>,<dbl>
1,AL,2000,4.4857
2,AL,2001,4.8915
3,AL,2002,4.7619
4,AL,2003,4.4333
5,AL,2004,6.3542
6,AL,2005,6.3330


## DEA: Retail Drug Transactions

In [5]:
# Load Data
dea_df <- read.csv(here("data", "DEA_Clean.csv"))
dea_df <- dea_df %>% filter(state != "DC")
head(dea_df)
dea_state <- dea_df

,state,year,hydro_gms,oxy_gms,fent_gms
,<chr>,<int>,<dbl>,<dbl>,<dbl>
1,AK,2000,27018.40,74395.72,613.2600
2,AK,2001,32213.77,110300.16,613.2600
3,AK,2002,35524.00,106095.60,722.8675
4,AK,2003,38834.23,101891.03,832.4750
5,AK,2004,42144.46,97686.46,942.0825
6,AK,2005,45454.69,93481.90,1051.6900


In [6]:
# Aggregate to US Level
dea_us = dea_df %>%
    group_by(year) %>%
    summarise(
        hydro_gms = sum(hydro_gms, na.rm=TRUE),
        oxy_gms = sum(oxy_gms, na.rm = TRUE),
        fent_gms = sum(fent_gms, na.rm = TRUE)
    )
head(dea_us)

year,hydro_gms,oxy_gms,fent_gms
<int>,<dbl>,<dbl>,<dbl>
2000,14530961,15233461,185509.8
2001,15555319,19851864,185509.8
2002,18118626,22518655,235870.3
2003,20681932,25185446,286230.9
2004,23245239,27852236,336591.5
2005,25808546,30519027,386952.1


## University of Kentucky Center for Poverty Research (UKCPR) National Welfare Data

In [7]:
uk_raw <- read.csv(here("data", "UKCPR_Raw.csv"))
uk_raw <- uk_raw %>% filter(state_name != "DC")
head(uk_raw)

,state_name,state,state_fips,year,Population,Employment,Unemployment,Unemployment.rate,Marginally.Food.Insecure,Food.Insecure,⋯,SSI.recipients..Disabled,Medicaid.beneficiaries,WIC.participation,NSLP.Free.Participation,NSLP.Reduced.Participation,NSLP.Total.Participation,SBP.Free.Participation,SBP.Reduced.Participation,SBP.Total.Participation,Fraction.of.the.Year.ACA.Expansion
,<chr>,<int>,<int>,<int>,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,AL,1,1,1980,3893888,1521183,148106,8.9,NA,NA,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,0
2,AK,2,2,1980,401851,169397,18008,9.6,NA,NA,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,0
3,AZ,3,4,1980,2718215,1146371,81630,6.6,NA,NA,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,0
4,AR,4,5,1980,2286435,922894,75386,7.6,NA,NA,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,0
5,CA,5,6,1980,23667902,10787673,791379,6.8,NA,NA,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,0
6,CO,6,8,1980,2889964,1405381,86267,5.8,NA,NA,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,0


In [8]:
uk_raw <- uk_raw %>%
    filter(
        year >= 2000 & year <= 2016,
    )
uk_raw <- uk_raw[, c("state_name", "year", "Population", "Unemployment",
    "Number.of.Poor..thousands.", "Gross.State.Product", "Food.Stamp.SNAP.Recipients",
    "Medicaid.beneficiaries", "State.Minimum.Wage")]

uk_raw <- uk_raw %>% 
    rename(
        state = state_name,
        year = year,
        pop = Population,
        unempl_pop = Unemployment,
        poverty_pop = Number.of.Poor..thousands.,
        gsp = Gross.State.Product,
        snap_pop = Food.Stamp.SNAP.Recipients,
        medicaid_pop = Medicaid.beneficiaries,
        min_wage = State.Minimum.Wage
    )

head(uk_raw)

,state,year,pop,unempl_pop,poverty_pop,gsp,snap_pop,medicaid_pop,min_wage
,<chr>,<int>,<int>,<int>,<int>,<dbl>,<dbl>,<int>,<dbl>
1,AL,2000,4452173,97629,583,120132.9,396057,564926,5.15
2,AK,2000,627963,20365,47,26815.8,37524,83161,5.65
3,AZ,2000,5160586,99302,607,164609.9,259006,483993,5.15
4,AR,2000,2678588,53606,436,68740.4,246572,392018,5.15
5,CA,2000,33987977,834629,4294,1366166.5,1831698,6168816,6.25
6,CO,2000,4326921,65107,425,180693.3,155948,278969,5.15


In [9]:
# Calculate Rates for State-Level Data
uk_state <- uk_raw %>%
    group_by(year) %>%
    mutate(
        unempl_rate = (unempl_pop / pop) * 100000,
        poverty_rate = (poverty_pop / pop) * 100000,
        snap_rate = (snap_pop / pop) * 100000,
        medicaid_rate = (medicaid_pop / pop) * 100000,
        gsp_per_cap = (gsp / pop) * 100000
    ) %>%
    filter(state != "DC")
uk_state <- uk_state[, c("state", "year", "pop", "unempl_rate", "poverty_rate", "snap_rate", "medicaid_rate", "gsp_per_cap", "min_wage")]
head(uk_state)

state,year,pop,unempl_rate,poverty_rate,snap_rate,medicaid_rate,gsp_per_cap,min_wage
<chr>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
AL,2000,4452173,2192.839,13.094729,8895.813,12688.770,2698.298,5.15
AK,2000,627963,3243.025,7.484517,5975.511,13242.978,4270.283,5.65
AZ,2000,5160586,1924.239,11.762230,5018.926,9378.644,3189.752,5.15
AR,2000,2678588,2001.278,16.277233,9205.298,14635.248,2566.292,5.15
CA,2000,33987977,2455.660,12.633879,5389.253,18149.995,4019.558,6.25
CO,2000,4326921,1504.696,9.822227,3604.133,6447.287,4176.025,5.15


In [11]:
# Aggregate to the U.S. level, weighted averages
uk_us <- uk_raw %>%
    group_by(year) %>%
    summarize(
        state = "United States",
        total_pop = sum(pop, na.rm = TRUE),
        unempl_rate = (sum(unempl_pop, na.rm = TRUE) / total_pop) * 100000,
        poverty_rate = (sum(poverty_pop, na.rm = TRUE) / total_pop * 100000),
        snap_rate = (sum(snap_pop, na.rm = TRUE) / total_pop) * 100000,
        medicaid_rate = (sum(medicaid_pop, na.rm = TRUE) / total_pop) * 100000,
        min_wage = sum(min_wage * pop, na.rm=TRUE) / total_pop,
        gsp = sum(gsp, na.rm = TRUE)
    )

uk_us <- uk_us[, c("year", "total_pop", "unempl_rate", "poverty_rate", "snap_rate", "medicaid_rate", "min_wage", "gsp")]
uk_us$gsp_per_cap = (uk_us$gsp / uk_us$total_pop) * 100000
uk_us <- uk_us[, c("year", "total_pop", "unempl_rate", "poverty_rate", "snap_rate", "medicaid_rate", "min_wage", "gsp_per_cap")]
head(uk_us)

year,total_pop,unempl_rate,poverty_rate,snap_rate,medicaid_rate,min_wage,gsp_per_cap
<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
2000,281590365,2016.930,11.18611,6044.427,12243.88,5.109301,3595.793
2001,284394451,2387.119,11.53609,6049.317,13090.47,5.333432,3674.482
2002,287052035,2913.727,12.00828,6612.955,14278.60,5.405677,3761.950
2003,289539431,3002.885,12.35341,7301.457,15121.85,5.426027,3906.651
2004,292237544,2767.105,12.62808,8120.057,15534.92,5.452147,4125.825
2005,294949463,2568.643,12.48926,8678.583,15889.26,5.614247,4364.135


In [12]:
# Join Data on Year
df_final_us = nchs_us %>%
    left_join(dea_us, by = "year") %>%
    left_join(uk_us, by = "year")

# Calculate Drug Supply per 100k people
df_final_us <- df_final_us %>%
    mutate(
        hydro_gms = (hydro_gms / total_pop) * 100000,
        oxy_gms = (oxy_gms / total_pop) * 100000,
        fent_gms = (fent_gms / total_pop) * 100000
    )

head(df_final_us)

,year,sex,age_group,age_group_detail,race,death_rate,hydro_gms,oxy_gms,fent_gms,total_pop,unempl_rate,poverty_rate,snap_rate,medicaid_rate,min_wage,gsp_per_cap
,<int>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,2016,Male,0-14,0–14,Hispanic,0.1527,91216.074,6326016.917,251436.41157,322255496,2389.180,12.56922,13656.855,22997.33,8.157172,5741.554
2,2012,Female,0-14,0–14,Hispanic,0.0686,552156.154,302736.979,314521.00164,313196066,3986.529,14.80830,14814.800,18975.03,7.448500,5099.413
3,2000,Female,0-14,0–14,Hispanic,0.0584,5160.319,5409.795,65.87930,281590365,2016.930,11.18611,6044.427,12243.88,5.109301,3595.793
4,2001,Female,0-14,0–14,Hispanic,0.0930,5469.628,6980.398,65.22974,284394451,2387.119,11.53609,6049.317,13090.47,5.333432,3674.482
5,2002,Female,0-14,0–14,Hispanic,0.1256,6311.966,7844.799,82.16989,287052035,2913.727,12.00828,6612.955,14278.60,5.405677,3761.950
6,2003,Female,0-14,0–14,Hispanic,0.1562,7143.045,8698.451,98.85731,289539431,3002.885,12.35341,7301.457,15121.85,5.426027,3906.651


In [13]:
head(nchs_state)
head(dea_state)
head(uk_state)

dim(nchs_state)
dim(dea_state)
dim(uk_state)

,state,year,death_rate
,<chr>,<int>,<dbl>
1,AL,2000,4.4857
2,AL,2001,4.8915
3,AL,2002,4.7619
4,AL,2003,4.4333
5,AL,2004,6.3542
6,AL,2005,6.3330


,state,year,hydro_gms,oxy_gms,fent_gms
,<chr>,<int>,<dbl>,<dbl>,<dbl>
1,AK,2000,27018.40,74395.72,613.2600
2,AK,2001,32213.77,110300.16,613.2600
3,AK,2002,35524.00,106095.60,722.8675
4,AK,2003,38834.23,101891.03,832.4750
5,AK,2004,42144.46,97686.46,942.0825
6,AK,2005,45454.69,93481.90,1051.6900


state,year,pop,unempl_rate,poverty_rate,snap_rate,medicaid_rate,gsp_per_cap,min_wage
<chr>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
AL,2000,4452173,2192.839,13.094729,8895.813,12688.770,2698.298,5.15
AK,2000,627963,3243.025,7.484517,5975.511,13242.978,4270.283,5.65
AZ,2000,5160586,1924.239,11.762230,5018.926,9378.644,3189.752,5.15
AR,2000,2678588,2001.278,16.277233,9205.298,14635.248,2566.292,5.15
CA,2000,33987977,2455.660,12.633879,5389.253,18149.995,4019.558,6.25
CO,2000,4326921,1504.696,9.822227,3604.133,6447.287,4176.025,5.15


[1] 850   3

[1] 850   5

[1] 850   9

In [14]:
# Join Data on Year, State
df_final_state = nchs_state %>%
    left_join(dea_state, by = c("year", "state")) %>%
    left_join(uk_state, by = c("year", "state"))

# Calculate Drug Supply per 100k Population
df_final_state <- df_final_state %>%
    mutate(
        hydro_gms = (hydro_gms/pop) * 100000,
        oxy_gms = (oxy_gms/pop) * 100000,
        fent_gms = (fent_gms/pop) * 100000
    )

df_final_state = df_final_state[, c("year", "state", "pop", "hydro_gms", "oxy_gms", "fent_gms",
    "unempl_rate", "poverty_rate", "snap_rate", "medicaid_rate", "gsp_per_cap", "min_wage", "death_rate")]
head(df_final_state)

,year,state,pop,hydro_gms,oxy_gms,fent_gms,unempl_rate,poverty_rate,snap_rate,medicaid_rate,gsp_per_cap,min_wage,death_rate
,<int>,<chr>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,2000,AL,4452173,10652.63,6716.904,62.74487,2192.839,13.09473,8895.813,12688.77,2698.298,5.15,4.4857
2,2001,AL,4467634,11269.21,7735.814,62.52773,2428.959,15.57872,9206.036,15086.46,2753.925,5.15,4.8915
3,2002,AL,4480089,12882.26,8232.095,83.42898,2773.204,14.28543,9900.406,16452.60,2859.707,5.15,4.7619
4,2003,AL,4503491,14451.14,8704.415,103.96102,2830.982,14.72191,10482.224,16606.21,2974.788,5.15,4.4333
5,2004,AL,4530729,15990.25,9164.086,124.17555,2683.211,16.84056,10982.581,17265.46,3242.010,5.15,6.3542
6,2005,AL,4569805,17465.61,9593.347,143.77506,2105.867,16.41208,12223.629,17692.77,3422.538,5.15,6.3330


In [15]:
# Export final dataset (state level)
write.csv(df_final_state, here("data", "death_rate.csv"), row.names = FALSE)